In [13]:
import re
import contractions
import pickle
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import nltk

In [14]:
#nltk.download('stopwords')
#nltk.download('punkt_tab')
#nltk.download('averaged_perceptron_tagger_eng')
#nltk.download('wordnet')

In [15]:
stop_words = set(stopwords.words("english")) - {
    "no",
    "not",
    "nor",
    "neither",
    "never",
    "cannot"
}

lemmatizer = WordNetLemmatizer()


def preprocess_custom_review(text):

    # Remove HTML
    text = re.sub(r"<.*?>", " ", text)

    # Lowercase
    text = text.lower()

    # Expand contractions
    text = contractions.fix(text)

    # Remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenization
    tokens = word_tokenize(text)

    # POS tagging
    pos_tags = pos_tag(tokens)

    # POS-aware lemmatization
    lemmatized = []

    for word, tag in pos_tags:

        if tag.startswith("J"):
            pos = wordnet.ADJ
        elif tag.startswith("V"):
            pos = wordnet.VERB
        elif tag.startswith("N"):
            pos = wordnet.NOUN
        elif tag.startswith("R"):
            pos = wordnet.ADV
        else:
            pos = wordnet.NOUN

        lemma = lemmatizer.lemmatize(word, pos)
        lemmatized.append(lemma)

    # Stopword removal
    final_tokens = [
        word for word in lemmatized
        if word not in stop_words
    ]

    # Convert tokens back to text
    processed_text = " ".join(final_tokens)

    return processed_text

In [16]:
with open("../model/tfidf_movie_review.pkl", "rb") as file:
    tfidf = pickle.load(file)

with open("../model/svm_model.pkl", "rb") as file:
    svm_model = pickle.load(file)

In [17]:
custom_reviews = [
    "A decent, middle-of-the-road blockbuster that is entertaining to watch once but ultimately forgettable.",
    "A major disappointment that relies too heavily on special effects rather than delivering a coherent script.",
    "Even a talented, star-studded cast cannot rescue the clunky dialogue and sluggish pacing.",
    "Audiences are completely captivated by the breathtaking cinematography and the lead actor's powerhouse performance.",
    "A masterclass in genre-bending filmmaking.",
    "A modern masterpiece that perfectly balances heart, humor, and stunning visual storytelling.",
    "Viewers leave the cinema feeling deeply moved, describing it as an emotional rollercoaster that stays with you long after the credits roll.",
    "Fans of the original source material feel let down by the confusing narrative choices and lack of character depth.",
    "A spectacular visual feast paired with a brilliant script that keeps you hooked from start to finish.",
    "The characters were so poorly written that I found myself completely disconnected from the emotional core.",
    "The innovative sound design and atmospheric lighting created a deeply unsettling and perfect horror experience.",
    "The plot was completely predictable and formulaic, making the entire two hours feel like an absolute chore.",
    "An incredible cinematic achievement that completely shatters expectations and effortlessly redefines its genre.",
    "The acting felt incredibly stiff and artificial, preventing any real emotional connection to the core characters.",
    "A delightful and charming experience that is guaranteed to put a massive smile on your face from start to finish.",
    "It tries way too hard to be deep and philosophical but ultimately ends up being pretentious, shallow nonsense.",
    "A masterfully paced psychological thriller that keeps your heart racing up until the final shocking second.",
    "The visuals were undeniably stunning, but they could not save the film from its incredibly dull and lifeless script.",
    "A rare gem of a film that perfectly balances heavy, complex themes with moments of pure, unfiltered joy.",
    "I walked out of the theater completely unsatisfied with how poorly and lazily the main conflicts were resolved.",
    "The chemistry between the two leads is absolutely electric, effortlessly carrying the film through its minor narrative flaws."    
]

In [18]:
processed_custom_reviews = [
    preprocess_custom_review(review)
    for review in custom_reviews
]

processed_custom_reviews

['decent middle road blockbuster entertain watch ultimately forgettable',
 'major disappointment rely heavily special effect rather deliver coherent script',
 'even talented star stud cast not rescue clunky dialogue sluggish pacing',
 'audience completely captivate breathtaking cinematography lead actor powerhouse performance',
 'masterclass genre bending filmmaking',
 'modern masterpiece perfectly balance heart humor stun visual storytelling',
 'viewer leave cinema feeling deeply move describe emotional rollercoaster stay long credit roll',
 'fan original source material feel let confuse narrative choice lack character depth',
 'spectacular visual feast pair brilliant script keep hook start finish',
 'character poorly write find completely disconnect emotional core',
 'innovative sound design atmospheric lighting create deeply unsettling perfect horror experience',
 'plot completely predictable formulaic make entire two hour feel like absolute chore',
 'incredible cinematic achievemen

In [19]:
custom_tfidf = tfidf.transform(processed_custom_reviews)

print("Custom TF-IDF shape:", custom_tfidf.shape)

Custom TF-IDF shape: (21, 5000)


In [20]:
custom_predictions = svm_model.predict(custom_tfidf)

decision_scores = svm_model.decision_function(custom_tfidf)

In [21]:
custom_results = pd.DataFrame({
    "Review": custom_reviews,
    "Processed Review": processed_custom_reviews,
    "Prediction": [
        "Positive" if pred == 1 else "Negative"
        for pred in custom_predictions
    ],
    "Decision Score": decision_scores
})

custom_results

,Review,Processed Review,Prediction,Decision Score
0,"A decent, middle-of-the-road blockbuster that ...",decent middle road blockbuster entertain watch...,Negative,-0.577433
1,A major disappointment that relies too heavily...,major disappointment rely heavily special effe...,Negative,-1.003297
2,"Even a talented, star-studded cast cannot resc...",even talented star stud cast not rescue clunky...,Negative,-0.816254
3,Audiences are completely captivated by the bre...,audience completely captivate breathtaking cin...,Positive,0.259413
4,A masterclass in genre-bending filmmaking.,masterclass genre bending filmmaking,Positive,0.379192
5,A modern masterpiece that perfectly balances h...,modern masterpiece perfectly balance heart hum...,Positive,1.332705
6,"Viewers leave the cinema feeling deeply moved,...",viewer leave cinema feeling deeply move descri...,Positive,0.422772
7,Fans of the original source material feel let ...,fan original source material feel let confuse ...,Negative,-0.995420
8,A spectacular visual feast paired with a brill...,spectacular visual feast pair brilliant script...,Positive,0.730816
9,The characters were so poorly written that I f...,character poorly write find completely disconn...,Negative,-0.812075
